In [ ]:
!pip install -q --progress-bar off diffusers transformers accelerate google-generativeai Pillow torch pandas gdown

In [ ]:
import os
import pandas as pd
import gdown

# URLs do seu projeto
PROJECT_FOLDER_URL = "https://drive.google.com/drive/folders/1KErQ86D9QppwkIm4I46VY0SlBC7jJNZ-"
PROMPTS_SHEET_URL = "https://docs.google.com/spreadsheets/d/1GvyTi1A9yo4lE8_88Sng1IlDi3YEDFK7dEamXohHq2A/edit#gid=0"
DRAFTS_FOLDER_ID = "1DQNw83XSfKKS4XjfZFo6V4Fz05swxJTk"

# 1. Download da pasta de rascunhos (Drafts)
DRAFTS_DIR = "/content/drafts"
if not os.path.exists(DRAFTS_DIR):
    print("Baixando pasta de rascunhos do Google Drive...")
    gdown.download_folder(id=DRAFTS_FOLDER_ID, output=DRAFTS_DIR, quiet=False, use_cookies=False)

# 2. Leitura direta da planilha do Google Sheets
csv_url = PROMPTS_SHEET_URL.replace("/edit#gid=0", "/export?format=csv")
df_prompts = pd.read_csv(csv_url)

print(f"\nDataset carregado com sucesso! Total de itens: {len(df_prompts)}")
display(df_prompts.head())

In [ ]:
import json
import google.generativeai as genai
from PIL import Image

#Chave do Google AI Studio (https://aistudio.google.com/app/apikey?utm_source=gemini)
genai.configure(api_key="REDACTED_SET_YOUR_OWN_KEY")
vision_model = genai.GenerativeModel('gemini-1.5-pro-latest')

def extract_bounding_boxes_from_sketch(draft_image, prompt_text):
    """
    Identifica os objetos desenhados no rascunho e retorna suas bounding boxes
    normalizadas entre 0.0 e 1.0 no formato [ymin, xmin, ymax, xmax].
    """
    system_prompt = f"""
    You are an expert bounding-box detection system for generative layout models.
    Analyze the sketch image and the user text prompt: "{prompt_text}".

    Tasks:
    1. Locate each object drawn in the sketch that corresponds to the text prompt.
    2. Extract their bounding box coordinates normalized from 0.0 to 1.0 in [ymin, xmin, ymax, xmax] format.
    3. Return ONLY a raw JSON object with keys "phrases" and "boxes".

    JSON Output Format:
    {{
      "phrases": ["description of object 1", "description of object 2"],
      "boxes": [
        [ymin, xmin, ymax, xmax],
        [ymin, xmin, ymax, xmax]
      ]
    }}
    """

    response = vision_model.generate_content([system_prompt, draft_image])

    # Limpa e converte a resposta para JSON
    clean_text = response.text.replace("```json", "").replace("```", "").strip()
    return json.loads(clean_text)

In [ ]:
import torch
from diffusers import StableDiffusionGLIGENPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Pipeline GLIGEN nativo sobre o Stable Diffusion 1.5
pipe = StableDiffusionGLIGENPipeline.from_pretrained(
    "masterful/gligen-1-4-sd-v1-5",
    torch_dtype=DTYPE
).to(DEVICE)

In [ ]:
import os

OUTPUT_DIR = "/content/resultados_mgi"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Loop sobre o dataset de testes
for index, row in df_prompts.iterrows():
    # Ajuste o nome das colunas 'filename' e 'prompt' se o cabeçalho da sua planilha for diferente
    image_filename = row.get("filename", f"{index}.png")
    user_prompt = row.get("prompt", "A realistic photo")

    draft_path = os.path.join(DRAFTS_DIR, image_filename)

    if not os.path.exists(draft_path):
        print(f"[{index}] Rascunho não encontrado em: {draft_path}. Pulando...")
        continue

    print(f"\n==========================================")
    print(f"[{index + 1}/{len(df_prompts)}] Processando: {image_filename}")
    print(f"Prompt: {user_prompt}")

    # 1. Carrega o rascunho do usuário
    draft_img = Image.open(draft_path).convert("RGB").resize((512, 512))

    # 2. Extrai as Bounding Boxes via LLM
    try:
        layout_data = extract_bounding_boxes_from_sketch(draft_img, user_prompt)
        phrases = layout_data.get("phrases", [])
        boxes = layout_data.get("boxes", [])
        print(f" Caixas detectadas ({len(boxes)}): {phrases}")
    except Exception as e:
        print(f" Erro na extração do LLM para a imagem {image_filename}: {e}")
        continue

    # 3. Gera a nova imagem com o GLIGEN usando o layout e o prompt
    generator = torch.Generator(device=DEVICE).manual_seed(42)

    final_image = pipe(
        prompt=user_prompt,
        gligen_phrases=phrases,
        gligen_boxes=boxes,
        gligen_scheduled_sampling_beta=0.4,
        num_inference_steps=35,
        guidance_scale=7.5,
        generator=generator
    ).images[0]

    # 4. Salva a imagem gerada
    save_path = os.path.join(OUTPUT_DIR, f"gerado_{image_filename}")
    final_image.save(save_path)
    print(f" Salvo em: {save_path}")

print("\n Execução do experimento MGI concluída com sucesso!")